# 💾 DQ Framework — Notebook 4: Results Writer

**Purpose:** Defines `ResultsWriter` — persists DQ rule results to Delta tables,
writes alerts, and prints a formatted run summary to the notebook output.

This notebook is `%run` by the controller. Depends on `DQRuleResult`
which is defined in notebook 02 (already loaded by the controller before this one).

## Imports

In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType, IntegerType, StringType, StructField, StructType,
)

## Delta Table Schema

In [0]:
DQ_RESULTS_SCHEMA = StructType([
    StructField("run_id",         StringType(),  False),
    StructField("dataset_name",   StringType(),  False),
    StructField("rule_id",        StringType(),  False),
    StructField("rule_type",      StringType(),  False),
    StructField("description",    StringType(),  True),
    StructField("severity",       StringType(),  False),
    StructField("status",         StringType(),  False),   # PASS | FAIL | ERROR
    StructField("passed",         BooleanType(), False),
    StructField("row_count",      IntegerType(), True),
    StructField("details",        StringType(),  True),
    StructField("error",          StringType(),  True),
    StructField("run_timestamp",  StringType(),  False),
])

DQ_ALERTS_SCHEMA = StructType([
    StructField("run_id",       StringType(), False),
    StructField("dataset_name", StringType(), False),
    StructField("rule_id",      StringType(), False),
    StructField("severity",     StringType(), False),
    StructField("status",       StringType(), False),
    StructField("details",      StringType(), True),
    StructField("alert_ts",     StringType(), False),
])

## ResultsWriter Class

In [0]:
class ResultsWriter:
    """
    Persists DQ results to Delta tables and prints a run summary.

    Parameters
    ----------
    spark          : active SparkSession
    results_table  : fully-qualified Delta table name for all results
                     e.g. 'main.dq_framework.dq_results'
    alerts_table   : fully-qualified Delta table name for FAIL/ERROR alerts
                     e.g. 'main.dq_framework.dq_alerts'

    The tables are created automatically on first use (CREATE TABLE IF NOT EXISTS).
    """

    def __init__(self, spark, results_table, alerts_table):
        self.spark         = spark
        self.results_table = results_table
        self.alerts_table  = alerts_table
        self.run_id        = datetime.utcnow().strftime("run_%Y%m%d_%H%M%S")
        self._ensure_tables()

    # ─────────────────────────────────────────────────────────────────────────
    def _ensure_tables(self):
        """Create results and alerts Delta tables if they don't already exist."""
        results_schema_ddl = ", ".join(
            f"{f.name} {f.dataType.simpleString()}"
            for f in DQ_RESULTS_SCHEMA.fields
        )
        self.spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.results_table} (
                {results_schema_ddl}
            )
            USING DELTA
            COMMENT 'DQ Framework: per-rule execution results'
        """)

        self.spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {self.alerts_table} (
                run_id       STRING  NOT NULL,
                dataset_name STRING  NOT NULL,
                rule_id      STRING  NOT NULL,
                severity     STRING  NOT NULL,
                status       STRING  NOT NULL,
                details      STRING,
                alert_ts     STRING  NOT NULL
            )
            USING DELTA
            COMMENT 'DQ Framework: CRITICAL and WARNING failures for alerting'
        """)

    # ─────────────────────────────────────────────────────────────────────────
    def write(self, results):
        """
        Append a list of DQRuleResult objects to the results Delta table.
        Also writes any CRITICAL or WARNING failures to the alerts table.

        Parameters
        ----------
        results : list[DQRuleResult]
        """
        if not results:
            print("[RESULTS] No results to write.")
            return

        rows = [
            (
                self.run_id,
                r.dataset_name,
                r.rule_id,
                r.rule_type,
                r.description,
                r.severity,
                r.status,
                r.passed,
                r.row_count,
                r.details,
                r.error,
                r.run_timestamp,
            )
            for r in results
        ]
        df = self.spark.createDataFrame(rows, schema=DQ_RESULTS_SCHEMA)
        (df.write
           .format("delta")
           .mode("append")
           .option("mergeSchema", "true")
           .saveAsTable(self.results_table))

        # ── Write alerts for CRITICAL and WARNING failures ─────────────────────
        alert_rows = [
            (
                self.run_id,
                r.dataset_name,
                r.rule_id,
                r.severity,
                r.status,
                r.details or r.error,
                r.run_timestamp,
            )
            for r in results
            if not r.passed and r.severity in ("CRITICAL", "WARNING")
        ]
        if alert_rows:
            alert_df = self.spark.createDataFrame(alert_rows, schema=DQ_ALERTS_SCHEMA)
            (alert_df.write
                     .format("delta")
                     .mode("append")
                     .saveAsTable(self.alerts_table))

    # ─────────────────────────────────────────────────────────────────────────
    def print_summary(self, results):
        """
        Print a formatted run summary to the notebook cell output.
        Includes per-dataset breakdown and a list of all failures.
        """
        total    = len(results)
        passed   = sum(1 for r in results if r.passed)
        failed   = sum(1 for r in results if not r.passed and not r.error)
        errored  = sum(1 for r in results if r.error)
        critical = sum(1 for r in results if not r.passed and r.severity == "CRITICAL")

        sep = "=" * 72
        print(f"\n{sep}")
        print(f"  DQ RUN SUMMARY   run_id = {self.run_id}")
        print(sep)
        print(f"  Total rules   : {total}")
        print(f"  ✅ Passed      : {passed}")
        print(f"  ❌ Failed      : {failed}")
        print(f"  💥 Errors      : {errored}")
        print(f"  🔴 Critical    : {critical}")
        print(sep)

        # Per-dataset summary
        datasets = sorted({r.dataset_name for r in results})
        for ds in datasets:
            ds_res  = [r for r in results if r.dataset_name == ds]
            ds_pass = sum(1 for r in ds_res if r.passed)
            ds_tot  = len(ds_res)
            ds_crit = sum(1 for r in ds_res if not r.passed and r.severity == "CRITICAL")
            icon    = "✅" if ds_pass == ds_tot else ("🔴" if ds_crit else "⚠️ ")
            crit_tag = f"  [{ds_crit} CRITICAL]" if ds_crit else ""
            print(f"  {icon} {ds:<35} {ds_pass}/{ds_tot} passed{crit_tag}")

        print(sep)

        # List every failure
        failures = [r for r in results if not r.passed]
        if failures:
            print("\n  FAILURES:")
            for r in failures:
                icon = "🔴" if r.severity == "CRITICAL" else ("⚠️ " if r.severity == "WARNING" else "ℹ️ ")
                print(f"    {icon} [{r.severity:<8}] {r.dataset_name}.{r.rule_id}")
                msg = r.details or r.error
                if msg:
                    # Wrap long lines for readability
                    for line in msg.split("\n"):
                        print(f"             {line}")
        print(sep + "\n")

### ✅ Results writer notebook loaded

Defines: `ResultsWriter`, `DQ_RESULTS_SCHEMA`, `DQ_ALERTS_SCHEMA`